# RL Intuition and PPO with a Library

This notebook builds intuition for reinforcement learning from the ground up,
starting with a multi-armed bandit and working up to training a PPO agent on
CartPole using Stable Baselines3.

In [ ]:
# !pip install stable-baselines3[extra] gymnasium
import random

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy

## Exploration vs. Exploitation

The core tension in RL: should the agent try something new (explore),
or repeat what worked before (exploit)?

A multi-armed bandit is the simplest environment to study this:
there are N slot machines (arms), each paying out a random reward with
a fixed but unknown mean. The goal is to maximize total reward over many pulls.

**Pure exploitation**: always pull the arm with the highest estimated mean.
Gets stuck if early estimates are wrong.

**Pure exploration**: pull randomly. Wastes pulls on arms you already know are bad.

**Epsilon-greedy**: with probability epsilon, explore (pull randomly);
otherwise, exploit (pull the best known arm). Simple but effective.

In [ ]:
class Bandit:
    def __init__(self, n_arms=5, seed=42):
        rng = np.random.default_rng(seed)
        self.means = rng.uniform(0, 1, n_arms)  # true reward means, unknown to agent
        self.best_arm = np.argmax(self.means)

    def pull(self, arm):
        return np.random.normal(self.means[arm], 0.5)  # noisy reward


def run_bandit(bandit, epsilon, n_steps=1000, seed=0):
    np.random.seed(seed)
    n_arms = len(bandit.means)
    estimates = np.zeros(n_arms)
    counts = np.zeros(n_arms)
    total_rewards = []
    cumulative = 0

    for _ in range(n_steps):
        if np.random.rand() < epsilon:
            arm = np.random.randint(n_arms)  # explore
        else:
            arm = np.argmax(estimates)        # exploit

        reward = bandit.pull(arm)
        counts[arm] += 1
        estimates[arm] += (reward - estimates[arm]) / counts[arm]  # running mean
        cumulative += reward
        total_rewards.append(cumulative)

    return total_rewards


bandit = Bandit(n_arms=5)
print("True arm means:", [f"{m:.3f}" for m in bandit.means])
print(f"Best arm: {bandit.best_arm} (mean={bandit.means[bandit.best_arm]:.3f})")

fig, ax = plt.subplots(figsize=(9, 4))
for eps, label in [(0.0, "pure exploit (eps=0)"), (0.1, "eps=0.1"), (0.5, "eps=0.5"), (1.0, "pure explore (eps=1)")]:
    rewards = run_bandit(bandit, epsilon=eps)
    ax.plot(rewards, label=label)

ax.set_xlabel("Steps")
ax.set_ylabel("Cumulative reward")
ax.set_title("Epsilon-greedy on a 5-arm bandit")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Policy and Value Function

Moving beyond bandits, in a sequential decision problem the agent's choice at
each step affects future states and future rewards.

**Policy** (pi): a function from state to action (or a distribution over actions).
The simplest policy is random: choose uniformly from all valid actions.

**Value function** V(s): the expected total discounted reward the agent will
receive starting from state s, following policy pi.

```
V(s) = E[r_t + gamma * r_{t+1} + gamma^2 * r_{t+2} + ...]
```

Gamma (the discount factor) controls how much the agent values future rewards.
A gamma of 0.9 means a reward 10 steps away is worth 0.9^10 = 0.35 of an immediate reward.

We estimate the value function using Monte Carlo rollouts: run the policy many times
from each state and average the returns.

In [ ]:
# Simple gridworld for policy/value demo
GRID = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 0, 2],  # 2 = goal
])

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
ROWS, COLS = GRID.shape
GOAL = (4, 4)


def step(state, action):
    r, c = state
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc
    if 0 <= nr < ROWS and 0 <= nc < COLS and GRID[nr, nc] != 1:
        state = (nr, nc)
    reward = 10.0 if state == GOAL else -0.1
    done = (state == GOAL)
    return state, reward, done


def random_policy(state):
    return random.randint(0, 3)


def monte_carlo_value(policy, n_rollouts=200, max_steps=50, gamma=0.95):
    value = np.zeros((ROWS, COLS))
    counts = np.zeros((ROWS, COLS))

    for _ in range(n_rollouts):
        start = (random.randint(0, ROWS-1), random.randint(0, COLS-1))
        if GRID[start] == 1:
            continue

        state = start
        rewards = []
        states_visited = [state]

        for _ in range(max_steps):
            action = policy(state)
            state, reward, done = step(state, action)
            rewards.append(reward)
            states_visited.append(state)
            if done:
                break

        G = 0
        for i in range(len(rewards) - 1, -1, -1):
            G = rewards[i] + gamma * G
            s = states_visited[i]
            value[s] += G
            counts[s] += 1

    mask = counts > 0
    value[mask] /= counts[mask]
    return value


V = monte_carlo_value(random_policy)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(V, cmap="coolwarm", interpolation="nearest")
plt.colorbar(im, ax=ax)
# overlay wall markers
for r in range(ROWS):
    for c in range(COLS):
        if GRID[r, c] == 1:
            ax.text(c, r, "X", ha="center", va="center", fontsize=14, color="black")
        else:
            ax.text(c, r, f"{V[r,c]:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Value function (random policy, gamma=0.95)")
ax.axis("off")
plt.tight_layout()
plt.show()

## PPO at a Working Level

Policy gradient methods train a neural network that directly outputs an action distribution.
The key question is: how do you update the policy without making it worse?

PPO (Proximal Policy Optimization) answers this with a **clipped objective**.
It measures how much the new policy deviates from the old one using the probability ratio:

```
r(theta) = pi_new(a | s) / pi_old(a | s)
```

If the ratio is too far from 1 (the policy changed too much), the gradient is clipped.
This prevents large, destabilizing updates while still allowing the policy to improve.

In practice you use a library. PPO is one of the most reliable algorithms for
continuous and discrete control: it's not the sample-efficient, but it's stable
and easy to tune.

In [ ]:
# CartPole-v1: balance a pole on a cart by moving left/right
# Reward: +1 for every timestep the pole stays upright
# Max episode length: 500 steps -> max reward = 500

env = gym.make("CartPole-v1")
eval_env = gym.make("CartPole-v1")

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="/tmp/ppo_cartpole",
    log_path="/tmp/ppo_cartpole",
    eval_freq=5000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=0,
)

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    verbose=0,
)

model.learn(total_timesteps=50_000, callback=eval_callback, progress_bar=True)
print("Training complete.")

In [ ]:
import numpy as np

# Load the evaluation log and plot reward over training
try:
    log = np.load("/tmp/ppo_cartpole/evaluations.npz")
    timesteps = log["timesteps"]
    mean_rewards = log["results"].mean(axis=1)
    std_rewards = log["results"].std(axis=1)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(timesteps, mean_rewards, label="Mean reward")
    ax.fill_between(
        timesteps,
        mean_rewards - std_rewards,
        mean_rewards + std_rewards,
        alpha=0.3,
        label="+/- 1 std",
    )
    ax.axhline(500, color="gray", linestyle="--", label="Max possible (500)")
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Episode reward")
    ax.set_title("PPO on CartPole-v1")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("Log file not found. Training may not have saved evaluations yet.")

## Hands-On: Save, Load, and Evaluate

In [ ]:
# Save the trained model
model.save("/tmp/ppo_cartpole_final")
print("Model saved.")

# Load it back
loaded_model = PPO.load("/tmp/ppo_cartpole_final", env=eval_env)
print("Model loaded.")

In [ ]:
# Evaluate over 10 episodes
mean_reward, std_reward = evaluate_policy(loaded_model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"Mean reward over 10 episodes: {mean_reward:.1f} +/- {std_reward:.1f}")

# For CartPole-v1, a reward near 500 means the pole stayed upright for the full episode.
# A reward below 200 means the agent is still struggling.
if mean_reward >= 450:
    print("Agent is reliably solving CartPole.")
elif mean_reward >= 200:
    print("Agent is improving but not yet reliable. Consider training longer.")
else:
    print("Agent is still learning. Training for more timesteps should help.")

In [ ]:
# Watch one episode
obs, _ = eval_env.reset()
total_reward = 0
for step_num in range(500):
    action, _ = loaded_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = eval_env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"Episode lasted {step_num + 1} steps, total reward = {total_reward}")

## From SB3 to Scratch: Building PPO Yourself

The sections above used Stable Baselines3, which hides the internals. Now we
build every component from scratch so you can see exactly what PPO does:

1. A neural network policy (and combined actor-critic)
2. Rollout collection
3. Generalised Advantage Estimation (GAE)
4. The clipped surrogate loss
5. A full training loop

We use **CartPole-v1** as the environment: 4-dimensional observations, 2 discrete
actions. It is small enough to train in minutes on CPU while still showing a clear
learning curve.

In [ ]:
# pip install gymnasium torch stable-baselines3[extra]  (if not already installed)
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ---- Environment -----------------------------------------------------------
env = gym.make("CartPole-v1")
obs, _ = env.reset()

print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)
print("Sample observation:", obs)
print("Obs shape:", env.observation_space.shape)
print("Num actions:", env.action_space.n)

## Policy Network and Actor-Critic

`PolicyNet` maps observations to action logits (unnormalised log-probabilities).
Sampling from `Categorical(logits=logits)` gives stochastic actions.

`ActorCritic` shares a trunk and adds two heads: one for action logits (the actor)
and one for a scalar state value (the critic). Sharing the trunk is common practice
because features useful for estimating value are also useful for selecting actions.

In [ ]:
class PolicyNet(nn.Module):
    """
    Simple MLP policy: obs -> action logits.

    Parameters
    ----------
    obs_dim    : dimension of the observation vector
    hidden_dim : number of units in each hidden layer
    act_dim    : number of discrete actions
    """
    def __init__(self, obs_dim, hidden_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, act_dim),
        )

    def forward(self, obs):
        """Return action logits (shape: [batch, act_dim])."""
        return self.net(obs)


class ActorCritic(nn.Module):
    """
    Combined actor-critic network with a shared trunk.

    Forward pass returns (logits, value).
    logits : [batch, act_dim]  -- fed to Categorical for action sampling
    value  : [batch]           -- scalar estimate of V(s)
    """
    def __init__(self, obs_dim, hidden_dim, act_dim):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.actor_head = nn.Linear(hidden_dim, act_dim)
        self.critic_head = nn.Linear(hidden_dim, 1)

    def forward(self, obs):
        features = self.trunk(obs)
        logits = self.actor_head(features)          # [B, act_dim]
        value  = self.critic_head(features).squeeze(-1)  # [B]
        return logits, value


# Sanity check
obs_dim = env.observation_space.shape[0]   # 4 for CartPole
act_dim = env.action_space.n               # 2 for CartPole

pnet = PolicyNet(obs_dim, hidden_dim=64, act_dim=act_dim)
ac   = ActorCritic(obs_dim, hidden_dim=64, act_dim=act_dim)

dummy_obs = torch.zeros(1, obs_dim)
logits_only = pnet(dummy_obs)
logits, value = ac(dummy_obs)
print("PolicyNet output shape  :", logits_only.shape)   # [1, 2]
print("ActorCritic logits shape:", logits.shape)         # [1, 2]
print("ActorCritic value shape :", value.shape)          # [1]

## Rollout Collection

`collect_rollout` runs the current policy in the environment for `n_steps` steps
and stores everything needed for a PPO update: states, actions, log-probabilities,
rewards, value estimates, and done flags.

All returned objects are PyTorch tensors for easy batched gradient computation.

In [ ]:
from torch.distributions import Categorical


def collect_rollout(env, ac_net, n_steps=512):
    """
    Collect a rollout of n_steps transitions.

    Parameters
    ----------
    env     : gymnasium environment
    ac_net  : ActorCritic network
    n_steps : number of environment steps to collect

    Returns
    -------
    states      : FloatTensor [n_steps, obs_dim]
    actions     : LongTensor  [n_steps]
    log_probs   : FloatTensor [n_steps]   -- log pi(a|s) under current policy
    rewards     : FloatTensor [n_steps]
    values      : FloatTensor [n_steps]   -- V(s) estimates
    dones       : FloatTensor [n_steps]   -- 1.0 if episode ended after this step
    ep_rewards  : list of float           -- total reward per completed episode
    """
    states_list    = []
    actions_list   = []
    log_probs_list = []
    rewards_list   = []
    values_list    = []
    dones_list     = []
    ep_rewards     = []

    obs, _ = env.reset()
    ep_reward = 0.0

    for _ in range(n_steps):
        obs_t = torch.FloatTensor(obs).unsqueeze(0)  # [1, obs_dim]

        with torch.no_grad():
            logits, value = ac_net(obs_t)
            dist  = Categorical(logits=logits)
            action = dist.sample()
            log_p  = dist.log_prob(action)

        next_obs, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        states_list.append(obs)
        actions_list.append(action.item())
        log_probs_list.append(log_p.item())
        rewards_list.append(float(reward))
        values_list.append(value.squeeze().item())
        dones_list.append(float(done))

        ep_reward += reward
        if done:
            ep_rewards.append(ep_reward)
            ep_reward = 0.0
            obs, _ = env.reset()
        else:
            obs = next_obs

    return (
        torch.FloatTensor(states_list),
        torch.LongTensor(actions_list),
        torch.FloatTensor(log_probs_list),
        torch.FloatTensor(rewards_list),
        torch.FloatTensor(values_list),
        torch.FloatTensor(dones_list),
        ep_rewards,
    )


# Quick test
ac_test = ActorCritic(obs_dim, hidden_dim=64, act_dim=act_dim)
states, actions, log_probs, rewards, values, dones, ep_rews = collect_rollout(env, ac_test, n_steps=256)
print("states shape   :", states.shape)
print("actions shape  :", actions.shape)
print("rewards mean   :", rewards.mean().item())
print("episodes done  :", len(ep_rews))
if ep_rews:
    print("mean ep reward :", np.mean(ep_rews))

## Generalised Advantage Estimation (GAE)

The advantage `A_t` measures how much better action `a_t` was compared to what
the policy would do on average at state `s_t`. A positive advantage means the
action was better than expected; the policy should increase its probability.

GAE interpolates between TD(1) (Monte Carlo, high variance) and TD(0) (single-step,
high bias) using the parameter `lambda`:

```
delta_t = r_t + gamma * V(s_{t+1}) * (1 - done_t) - V(s_t)

A_t = delta_t + (gamma * lambda) * delta_{t+1} + (gamma * lambda)^2 * delta_{t+2} + ...
    = sum_{l=0}^{T-t} (gamma * lambda)^l * delta_{t+l}
```

- `lambda = 0`: pure TD(0), low variance, high bias
- `lambda = 1`: Monte Carlo, high variance, unbiased

`gamma = 0.99`, `lambda = 0.95` is a reliable default for most environments.

In [ ]:
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """
    Generalised Advantage Estimation.

    Parameters
    ----------
    rewards : FloatTensor [T]
    values  : FloatTensor [T]   -- V(s_t) for each step
    dones   : FloatTensor [T]   -- 1.0 if episode ended at step t
    gamma   : discount factor
    lam     : GAE lambda

    Returns
    -------
    advantages : FloatTensor [T]
    returns    : FloatTensor [T]  -- advantages + values (targets for critic)
    """
    T = len(rewards)
    advantages = torch.zeros(T)
    gae = 0.0

    # Bootstrap value after last step is 0 (or V(s_{T+1}) if not done)
    next_value = 0.0

    for t in reversed(range(T)):
        mask = 1.0 - dones[t].item()
        delta = rewards[t].item() + gamma * next_value * mask - values[t].item()
        gae   = delta + gamma * lam * mask * gae
        advantages[t] = gae
        next_value = values[t].item()

    returns = advantages + values
    return advantages, returns


# Demo: compute advantages for a random rollout
adv, ret = compute_gae(rewards, values, dones)
print("Advantages -- mean: {:.3f}  std: {:.3f}  min: {:.3f}  max: {:.3f}".format(
    adv.mean().item(), adv.std().item(), adv.min().item(), adv.max().item()))
print("Returns    -- mean: {:.3f}  std: {:.3f}".format(ret.mean().item(), ret.std().item()))

## PPO Loss

PPO combines three terms:

```
L_total = -L_clip + vf_coef * L_vf - ent_coef * H

# Clipped policy loss (we maximise this, so we negate it as a loss):
ratio = exp(log_prob_new - log_prob_old)
L_clip = E[ min(ratio * A, clip(ratio, 1-eps, 1+eps) * A) ]

# Value function loss (mean squared error):
L_vf = E[ (V(s) - returns)^2 ]

# Entropy bonus (encourages exploration):
H = E[ -sum_a pi(a|s) log pi(a|s) ]
```

The clip on the ratio stops the policy from taking steps that are too large.
If `ratio` drifts far from 1, the gradient is zeroed out for that sample.

In [ ]:
def ppo_loss(states, actions, old_log_probs, advantages, returns,
             ac_net, clip_eps=0.2, vf_coef=0.5, ent_coef=0.01):
    """
    Compute the PPO objective.

    Parameters
    ----------
    states        : FloatTensor [B, obs_dim]
    actions       : LongTensor  [B]
    old_log_probs : FloatTensor [B]  -- log pi_old(a|s), detached (no grad)
    advantages    : FloatTensor [B]  -- normalised GAE estimates
    returns       : FloatTensor [B]  -- targets for the value head
    ac_net        : ActorCritic network
    clip_eps      : PPO clipping epsilon (default 0.2)
    vf_coef       : weight of the value loss term
    ent_coef      : weight of the entropy bonus

    Returns
    -------
    total_loss : scalar tensor (to call .backward() on)
    info       : dict with policy_loss, value_loss, entropy
    """
    logits, values = ac_net(states)
    dist = Categorical(logits=logits)

    # New log-probabilities under the current policy
    new_log_probs = dist.log_prob(actions)

    # Probability ratio: pi_new / pi_old (in log space for stability)
    ratio = torch.exp(new_log_probs - old_log_probs)

    # Normalise advantages to zero mean, unit std (stabilises training)
    adv_norm = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # Clipped surrogate policy objective
    surr1 = ratio * adv_norm
    surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv_norm
    policy_loss = -torch.min(surr1, surr2).mean()   # we minimise the negation

    # Value function loss (MSE between predicted value and discounted returns)
    value_loss = ((values - returns) ** 2).mean()

    # Entropy bonus: higher entropy = more exploratory policy
    entropy = dist.entropy().mean()

    total_loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

    return total_loss, {
        "policy_loss": policy_loss.item(),
        "value_loss":  value_loss.item(),
        "entropy":     entropy.item(),
        "ratio_mean":  ratio.mean().item(),
    }

## Full PPO Training Loop

Each iteration:
1. Collect `n_steps` transitions with the current policy.
2. Compute GAE advantages and returns.
3. Run `n_epochs` passes over the data in random minibatches of size `batch_size`.
4. Log mean episode reward every 10 iterations.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# Hyperparameters
N_ITER      = 100    # number of PPO update iterations
N_STEPS     = 512    # rollout length per iteration
N_EPOCHS    = 4      # update epochs per iteration
BATCH_SIZE  = 128    # minibatch size
LR          = 3e-4
GAMMA       = 0.99
LAM         = 0.95
CLIP_EPS    = 0.2
VF_COEF     = 0.5
ENT_COEF    = 0.01

train_env  = gym.make("CartPole-v1")

scratch_ac  = ActorCritic(obs_dim, hidden_dim=64, act_dim=act_dim)
optimizer   = optim.Adam(scratch_ac.parameters(), lr=LR)

mean_rewards_log = []   # mean episode reward per iteration
entropy_log      = []   # policy entropy per iteration

for iteration in range(N_ITER):
    # ---- Collect rollout ------------------------------------------------
    (states_, actions_, log_probs_, rewards_,
     values_, dones_, ep_rews) = collect_rollout(train_env, scratch_ac, N_STEPS)

    with torch.no_grad():
        advantages_, returns_ = compute_gae(rewards_, values_, dones_, GAMMA, LAM)

    # ---- PPO update ------------------------------------------------------
    iter_entropy = []
    idx = torch.randperm(N_STEPS)

    for epoch in range(N_EPOCHS):
        idx = torch.randperm(N_STEPS)
        for start in range(0, N_STEPS, BATCH_SIZE):
            mb = idx[start:start + BATCH_SIZE]
            loss, info = ppo_loss(
                states_[mb], actions_[mb],
                log_probs_[mb].detach(),
                advantages_[mb], returns_[mb],
                scratch_ac, CLIP_EPS, VF_COEF, ENT_COEF,
            )
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(scratch_ac.parameters(), 0.5)
            optimizer.step()
            iter_entropy.append(info["entropy"])

    mean_ep_rew = float(np.mean(ep_rews)) if ep_rews else float(rewards_.mean())
    mean_rewards_log.append(mean_ep_rew)
    entropy_log.append(np.mean(iter_entropy))

    if (iteration + 1) % 10 == 0:
        print(f"Iter {iteration+1:3d}/{N_ITER}  "
              f"mean_ep_reward={mean_ep_rew:6.1f}  "
              f"entropy={entropy_log[-1]:.3f}")

print("Training complete.")

## Training Curve and Entropy Over Time

Two plots:
1. Mean episode reward per iteration -- should climb toward 500.
2. Policy entropy over training -- starts high (random policy) and collapses as
   the agent becomes confident. Low entropy late in training is normal and expected.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Reward curve
ax1.plot(mean_rewards_log, color="tab:blue")
ax1.axhline(500, color="gray", linestyle="--", alpha=0.5, label="Max (500)")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Mean episode reward")
ax1.set_title("Scratch PPO training curve (CartPole-v1)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Entropy curve
ax2.plot(entropy_log, color="tab:orange")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Policy entropy (nats)")
ax2.set_title("Entropy collapses as policy converges")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final mean reward:  {mean_rewards_log[-1]:.1f}")
print(f"Final entropy:      {entropy_log[-1]:.4f}")

## SB3 Comparison

Stable Baselines3's PPO is a well-tuned, production-grade implementation.
Running it alongside our scratch version lets you see whether both converge
to similar rewards. SB3 usually trains faster because it uses vectorised
environments and more careful hyperparameter defaults.

In [ ]:
from stable_baselines3 import PPO as SB3_PPO
from stable_baselines3.common.evaluation import evaluate_policy as sb3_eval

sb3_env  = gym.make("CartPole-v1")
sb3_model = SB3_PPO("MlpPolicy", sb3_env, verbose=0,
                    learning_rate=3e-4, n_steps=512,
                    batch_size=128, n_epochs=4)
sb3_model.learn(total_timesteps=N_ITER * N_STEPS)

# Evaluate both
sb3_mean, sb3_std = sb3_eval(sb3_model, gym.make("CartPole-v1"),
                              n_eval_episodes=10, deterministic=True)

# Scratch eval (run 10 episodes greedily)
scratch_rewards = []
eval_env2 = gym.make("CartPole-v1")
for _ in range(10):
    obs, _ = eval_env2.reset()
    ep_r = 0
    for _ in range(500):
        with torch.no_grad():
            logits, _ = scratch_ac(torch.FloatTensor(obs).unsqueeze(0))
            action = logits.argmax().item()
        obs, r, term, trunc, _ = eval_env2.step(action)
        ep_r += r
        if term or trunc:
            break
    scratch_rewards.append(ep_r)
scratch_mean = np.mean(scratch_rewards)
scratch_std  = np.std(scratch_rewards)

print(f"Scratch PPO:  {scratch_mean:.1f} +/- {scratch_std:.1f}")
print(f"SB3 PPO:      {sb3_mean:.1f} +/- {sb3_std:.1f}")
print(f"(Both trained on {N_ITER * N_STEPS:,} environment steps)")

## Exercise: Effect of Lambda on Advantage Variance

GAE's `lambda` parameter interpolates between TD(0) (`lambda=0`, low variance,
high bias) and Monte Carlo returns (`lambda=1`, high variance, unbiased).

Modify `compute_gae` to accept any `lam` value, run it with `lam=0.0` and
`lam=1.0` on the same rollout, and plot a histogram comparing the variance
of advantages under each setting.

In [ ]:
# Collect a fresh rollout to use for the comparison
(s_ex, a_ex, lp_ex, r_ex, v_ex, d_ex, _) = collect_rollout(
    gym.make("CartPole-v1"), scratch_ac, n_steps=512)

# YOUR CODE HERE
# 1. Compute advantages with lam=0.0 and lam=1.0 using compute_gae.
# 2. Plot histograms of both advantage distributions side by side.
# 3. Print the variance for each setting.

raise NotImplementedError